# SuperBook Wan VACE GPU Proof

This notebook is the zero-cost GPU proof for the isolated Python video engine. It uses the official **Wan2.1 VACE 1.3B Diffusers** reference-to-video path: one SuperBook scene image + semantic motion prompt → real MP4.

**Kaggle:** enable `GPU` in Notebook Settings. Kaggle now uses T4x2 as the supported accelerator after the P100 retirement. This notebook does not touch the SuperBook Flutter app.


In [ ]:
!nvidia-smi
!python -m pip install -q -U 'diffusers>=0.35.0' 'transformers>=4.49.0' 'accelerate>=1.2.0' safetensors 'pillow>=10.0.0' 'imageio[ffmpeg]>=2.36.0' ipywidgets


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Open Notebook Settings and select GPU.')
print('GPU 0:', torch.cuda.get_device_name(0))


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display
from pathlib import Path

uploader = FileUpload(accept='image/*', multiple=False, description='Upload scene image')
display(uploader)
print('Upload the actual SuperBook scene image, then run the next cell.')


In [ ]:
from PIL import Image

if not uploader.value:
    raise RuntimeError('No image uploaded yet.')
item = next(iter(uploader.value.values())) if isinstance(uploader.value, dict) else uploader.value[0]
name = item['name'] if isinstance(item, dict) else item.name
data = item['content'] if isinstance(item, dict) else item.content
IMAGE_PATH = Path('/kaggle/working') / name
IMAGE_PATH.write_bytes(bytes(data))
image = Image.open(IMAGE_PATH).convert('RGB')
print('Image:', IMAGE_PATH, image.size)
display(image)


## Real reference-to-video generation

VACE accepts the uploaded image as `reference_images`. This is not a slideshow or a browser animation. The model generates a continuous video sequence from the reference image and motion prompt.


In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanVACEPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

MODEL_ID = 'Wan-AI/Wan2.1-VACE-1.3B-diffusers'
print('Loading', MODEL_ID)
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.bfloat16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('Model loaded')


In [ ]:
MOTION_PROMPT = '''Cinematic storybook scene. Preserve the exact characters, faces, clothing, environment, lighting, furniture and composition from the reference image. The main character performs a natural, readable action matching the story: subtle breathing, shifts body weight, turns naturally toward the other character, and makes one deliberate hand gesture while speaking. The second character listens with small natural head and posture movement. Continuous physical motion, stable identity, coherent anatomy, gentle cinematic camera movement only. No new characters, no scene replacement, no camera cut, no morphing, no duplicated limbs, no text, no subtitles, no watermark.'''
NEGATIVE_PROMPT = '''static image, frozen pose, scene change, camera cut, new characters, duplicated characters, extra limbs, deformed hands, deformed face, morphing, text, subtitles, watermark, low quality, blurry, flicker'''
print(MOTION_PROMPT)


In [ ]:
from diffusers.utils import export_to_video

generator = torch.Generator(device='cuda').manual_seed(42)
result = pipe(
    prompt=MOTION_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    reference_images=[image],
    height=480,
    width=832,
    num_frames=81,
    num_inference_steps=20,
    guidance_scale=5.0,
    generator=generator,
)
frames = result.frames[0]
OUT = '/kaggle/working/superbook_wan_vace.mp4'
export_to_video(frames, OUT, fps=16)
print('Generated:', OUT)


In [ ]:
from IPython.display import Video, display
display(Video('/kaggle/working/superbook_wan_vace.mp4', embed=True))


## Gate

**PASS:** meaningful continuous character/body motion, preserved identity and scene continuity.

**FAIL:** camera drift only, severe face/limb corruption, scene replacement, or negligible motion.

If this passes, the same Python engine can be exposed through its FastAPI contract. The Flutter app remains untouched until the video proof passes.
